# Week 3 Day 5 - Lab 1: UnderCover in production

## Deploying everything for the handover

Today's build added the UnderCover Copilot, an agent for our claims handlers in a sidebar on the dashboard. It runs on AgentCore Runtime as a second runtime beside the claim agent, saves each conversation in Supabase and has an AgentCore Memory of its own, with a namespace for each handler. In this lab we deploy all of UnderCover to AWS, then have the handover conversation with the Copilot on the deployed dashboard.

## 6 parts:

PART 1: The new app

PART 2: Day 3's pieces

PART 3: Both Memories

PART 4: Both runtimes

PART 5: Pointing the app at both runtimes

PART 6: The handover conversation

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">You need Day 4's setup, with the AgentCore CLI at 1.0.0-rc.5, and today's build of the product, yours or <code>week3day5-end</code>, with <code>OPENROUTER_API_KEY</code>, <code>DATABASE_URL</code>, <code>VITE_SUPABASE_URL</code> and <code>UNDERCOVER_API_KEY</code> in your <code>.env</code> file. Keep a terminal open in the <code>undercover</code> folder for the CLI's commands.<br/><br/>The app and the context layer are inside AWS's free allowance for Lambda. The Gateway, the Policy, the two Memories and the secrets cost a fraction of a cent while they exist. Runtime charges only while a session is working, so the handover costs a cent or two on AWS and a few cents on OpenRouter. The CDK bootstrap's encryption key costs $1 a month while it exists. Part 5 reseeds your database, which starts the queue, the scoreboard and the configuration's versions afresh.<br/><br/>Everything stays up at the end of this lab, and Lab 2 deletes it all. If you stop partway, go straight to Lab 2.<br/><br/>If you're on the free route, <a href="week3day5_lab1free.ipynb">week3day5_lab1free.ipynb</a> is a guide to running the Copilot on Render beside the app.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Watching your AWS spend</h2>
            <span style="color:#118AB2;">Before you start, open a terminal in the <code>undercover</code> folder and run <code>uv run monitor.py</code>. It opens a page in your browser with what your AWS account has spent today and over the last 30 days, with and without your credits. AWS updates the figures up to three times a day.<br/><br/>For the official source of truth, please review Billing and Cost Management in your AWS console. Always have budgets with billing alerts, like the one from Day 1.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
import uuid
from pathlib import Path

import boto3
from dotenv import load_dotenv, set_key

import week3day4_aws as aws

## Connect to AWS

boto3 uses the access key that `aws configure` saved on Day 1, in us-east-1.

In [ ]:
load_dotenv(override=True)

session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

In [ ]:
lambda_client = session.client("lambda")
iam = session.client("iam")
agentcore_control = session.client("bedrock-agentcore-control")
agentcore_runtime = session.client("bedrock-agentcore")

In [ ]:
for name in ["OPENROUTER_API_KEY", "DATABASE_URL", "VITE_SUPABASE_URL", "UNDERCOVER_API_KEY"]:
    print(f"{name}: {'found' if os.getenv(name) else 'missing, please add it to .env'}")

API_KEY = os.environ["UNDERCOVER_API_KEY"]
# Supabase's transaction pooler, which the app on Lambda and both runtimes share
POOLED_DATABASE_URL = os.environ["DATABASE_URL"].replace(":5432/", ":6543/")

## Part 1: The new app

Today's build added the Copilot's routes to the app and its sidebar to the dashboard, so the app needs the new code. `week3day4_aws.py`, beside this notebook, holds Day 3's and Day 4's deployment code as functions. We build the dashboard, then update the function in place, which keeps its address. If you deleted the function on Day 4, this creates it again.

In [ ]:
!cd .. && npm run build

In [ ]:
app_environment = {
    "DATABASE_URL": POOLED_DATABASE_URL,
    "VITE_SUPABASE_URL": os.environ["VITE_SUPABASE_URL"],
    "UNDERCOVER_API_KEY": API_KEY,
}

app_url = aws.deploy_app(app_environment)
print(app_url)

## Part 2: Day 3's pieces

### The context layer

The context layer hasn't changed since Day 3, and the claim agent and the Copilot both use it. Deploying it again makes sure it's there.

In [ ]:
context_url = aws.deploy_context()
print(context_url)

### The Gateway and the Policy

The Gateway with the app and the context layer as its targets, the app's API key in AgentCore Identity, and the Policy enforced, with its forbid on any payment over $1,000. It takes about a minute.

The claim agent uses both targets. The Copilot uses only the context layer's tools on the Gateway: everything it does in the app, it does with the handler's own sign-in, and the app target would act as the agent.

In [ ]:
gateway = aws.create_gateway(app_url, context_url, API_KEY)
gateway_url = gateway["gatewayUrl"]
print(gateway_url)

## Part 3: Both Memories

### The claim agent's Memory

The claim agent's memory of each customer, with its Semantic and Summary strategies, as on Days 3 and 4. It takes about two and a half minutes.

In [ ]:
memory_id = aws.create_memory()

### The Copilot's Memory

The Copilot has a Memory of its own, apart from the claim agent's, because the claim agent's strategies would file everything under `/customers/{actorId}/`. In the Copilot's Memory the actor is the handler, taken from their sign-in, and the session is the conversation. Two strategies turn each conversation into notes under the handler's own namespace:

- **User Preference** keeps how the handler likes to work, such as how they want answers set out, in `/handlers/{actorId}/preferences/`
- **Semantic** keeps facts the handler mentions, such as the claims they're watching, in `/handlers/{actorId}/facts/`

It takes about two and a half minutes.

In [ ]:
COPILOT_MEMORY_NAME = "undercover_copilot_memory"

pages = agentcore_control.get_paginator("list_memories").paginate()
existing = [m["id"] for page in pages for m in page["memories"] if m["id"].startswith(COPILOT_MEMORY_NAME) and m["status"] != "DELETING"]

if existing:
    copilot_memory_id = existing[0]
else:
    copilot_memory_id = agentcore_control.create_memory(
        name=COPILOT_MEMORY_NAME,
        description="The UnderCover Copilot's memory of each handler",
        eventExpiryDuration=7,
        memoryStrategies=[
            {"userPreferenceMemoryStrategy": {"name": "handler_preferences", "namespaceTemplates": ["/handlers/{actorId}/preferences/"]}},
            {"semanticMemoryStrategy": {"name": "handler_facts", "namespaceTemplates": ["/handlers/{actorId}/facts/"]}},
        ],
        tags=aws.PROJECT_TAG,
    )["memory"]["id"]

status = agentcore_control.get_memory(memoryId=copilot_memory_id)["memory"]["status"]
while status == "CREATING":
    time.sleep(10)
    status = agentcore_control.get_memory(memoryId=copilot_memory_id)["memory"]["status"]

print(f"Memory {copilot_memory_id}: {status}")

## Part 4: Both runtimes

### What's in the repo

Please open `agentcore/agentcore.json` in your editor. Today's build added a second runtime, `copilot`, beside `agent`. Both are built from the same package with the `runtime` dependency group and share the project's three credentials. Each starts its own AgentCore app. `agentcore deploy` deploys the two together, both on V2.

The Copilot is an AgentCore app like the claim agent. Its one entrypoint takes a payload that names an operation, such as a turn of a conversation. It answers at once and works on the turn as an async task, so the session stays busy until the turn is done. On its first request it fetches its credentials from AgentCore Identity and opens its pool. The Copilot never uses the agent's API key: it calls the app with the sign-in of the handler it's working for.

The app sends every turn of a conversation with the conversation's id as the Runtime session id. While a conversation is active, its turns go to the same session. Once a session has been idle for the runtime's idle timeout it ends, and the next turn starts a new session under the same id, which picks the conversation up from its checkpoints in Supabase.

### The Claude Artifact that I generated unexpectedly!

Here is the link to the website that Claude made me. I don't know if this stays around forever, but give it a try. An interactive view of the graph in `graph.py`.. rather more fancy than I was looking for..

https://claude.ai/artifact/BrpwPKxuDwHSevJNUt4tJA

### The CDK project's packages

If you haven't installed them since Day 4, in your terminal, in the `undercover` folder, run:

`npm ci --prefix agentcore/cdk`

### The secrets and the settings

The CLI reads each credential's value from `agentcore/.env.local`, as on Day 4. Git ignores the file. The three values are the same as Day 4's, with the database on the transaction pooler.

The deploy also reads four settings from the same file and gives them to the runtimes as environment variables: the app's address, the Gateway's, the claim agent's Memory and the Copilot's Memory.

In [ ]:
LOCAL_ENV = Path("../agentcore/.env.local")
LOCAL_ENV.touch()

secrets = {
    "AGENTCORE_CREDENTIAL_OPENROUTER_API_KEY": os.environ["OPENROUTER_API_KEY"],
    "AGENTCORE_CREDENTIAL_UNDERCOVER_API_KEY": API_KEY,
    "AGENTCORE_CREDENTIAL_DATABASE_URL": POOLED_DATABASE_URL,
}
settings = {
    "UNDERCOVER_API": app_url,
    "UNDERCOVER_GATEWAY_URL": gateway_url,
    "UNDERCOVER_MEMORY_ID": memory_id,
    "UNDERCOVER_COPILOT_MEMORY_ID": copilot_memory_id,
}

for name, value in (secrets | settings).items():
    set_key(LOCAL_ENV, name, value)

print(f"Wrote {', '.join(secrets | settings)} to {LOCAL_ENV}")

### Deploying

In your terminal, in the `undercover` folder, run:

`agentcore deploy`

It takes about 10 minutes. Day 4's teardown deleted the CDK bootstrap, so the deploy makes it again first. Then it makes the three credentials in AgentCore Identity, and the stack `AgentCore-undercover-default` with both runtimes and their roles.

In [ ]:
agent_arn = aws.find_runtimes("undercover_default_agent")[0]["agentRuntimeArn"]
copilot_arn = aws.find_runtimes("undercover_default_copilot")[0]["agentRuntimeArn"]

print(agent_arn)
print(copilot_arn)

## Part 5: Pointing the app at both runtimes

Two changes to the app on Lambda. Its role may now invoke both runtimes, and nothing else on Runtime. And two settings tell it where to send the work: `UNDERCOVER_AGENT_RUNTIME` for the claim agent's work, and `UNDERCOVER_COPILOT_RUNTIME` for the Copilot's turns.

In [ ]:
invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "bedrock-agentcore:InvokeAgentRuntime",
            "Resource": [agent_arn, f"{agent_arn}/runtime-endpoint/*", copilot_arn, f"{copilot_arn}/runtime-endpoint/*"],
        }
    ],
}

iam.put_role_policy(RoleName=aws.APP_ROLE_NAME, PolicyName="undercover-app-runtime", PolicyDocument=json.dumps(invoke_policy))
print(f"{aws.APP_ROLE_NAME} may invoke both runtimes")

In [ ]:
app_environment["UNDERCOVER_AGENT_RUNTIME"] = agent_arn
app_environment["UNDERCOVER_COPILOT_RUNTIME"] = copilot_arn

lambda_client.update_function_configuration(FunctionName=aws.APP_FUNCTION, Environment={"Variables": app_environment})
lambda_client.get_waiter("function_updated_v2").wait(FunctionName=aws.APP_FUNCTION)
print(f"{aws.APP_FUNCTION} sends the agent's work and the Copilot's turns to Runtime")

### A fresh queue

We reseed, so all 26 claims are waiting and the Copilot starts with no conversations.

In [ ]:
!cd .. && npm run seed

### Calling each runtime

A health check, `{"operation": "health"}`, in a new session for each runtime. Each session gets ready before it answers: it fetches its secrets from Identity and opens its pool.

In [ ]:
def health_check(runtime_arn):
    """Send the health check to a runtime in a new session, and say how long it took."""
    started = time.time()
    response = agentcore_runtime.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn, runtimeSessionId=f"lab-{uuid.uuid4()}", payload=json.dumps({"operation": "health"})
    )
    reply = json.loads(response["response"].read())
    print(f"{time.time() - started:.1f} seconds: {reply}")


health_check(agent_arn)
health_check(copilot_arn)

## Part 6: The handover conversation

Open the deployed dashboard and sign in as Dana Whitfield, who can sign off over the limit and send decisions back. The Copilot's sidebar is open on the right of every page. Stay on the queue page, so you can watch the claims move beside the conversation.

In [ ]:
print(app_url)

In one conversation, ask the Copilot these five questions, one at a time:

1. **"Work the queue with 10 agents and tell me what needs me."** The Copilot starts a dispatch with Dana's sign-in, and the app starts it on the claim agent's runtime in a session of its own. The Copilot's own session waits for it in code, checking the app every few seconds, while the queue fills in beside the sidebar.
2. **"Why does CLM-... need me?"**, about one of the payouts it says are waiting for a sign-off. Watch the cards: the claim, the agent's steps on it and any context layer lookups, through the Gateway.
3. **"Send it back with a note"**, saying what to look at again. The app sends it back as Dana, and the claim agent picks the claim up in a new session.
4. **"Sign off on another one."** The Copilot picks another payout waiting for a sign-off, and its graph pauses before it pays, with the confirmation card in the sidebar. Press Sign off and pay. The app sends Dana's answer to the Copilot's runtime, which resumes the conversation from its checkpoint and signs off as Dana.
5. **"Are we meeting the success criteria we agreed?"** The Copilot runs an evaluation of the held-out set, waits for it and checks each criterion. This one takes a few minutes.

Runtime keeps each session's output in CloudWatch. In your terminal, `agentcore log runtime --name copilot --since 15m` shows the Copilot's logs, and `--name agent` the claim agent's.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Two runtimes in one conversation</h2>
            <span style="color:#118AB2;">Every turn went to the Copilot's runtime with the conversation's id as its session id, and kept working after the app had its reply. A turn soon after the last one reused the session, and a later one started a new session under the same id from the checkpoints in Supabase. The dispatch, the evaluation and the send back each ran on the claim agent's runtime in sessions of their own, started by the app. Between turns, everything either agent needs is in Supabase: the claims, the conversation's checkpoints and the steps the sidebar shows.
            </span>
        </td>
    </tr>
</table>

### What the Copilot remembers

Tell the Copilot how you like your answers, for example "I like the amounts first, then the reasons, in short bullet points". AgentCore Memory turns each turn into notes in about a minute and a half. Once it has, this cell lists every note in Dana's namespace. Her handler id, `HND-01`, is in `data/handlers.json`.

In [ ]:
DANA = "HND-01"

for kind in ["preferences", "facts"]:
    records = agentcore_runtime.list_memory_records(memoryId=copilot_memory_id, namespace=f"/handlers/{DANA}/{kind}/")
    for record in records["memoryRecordSummaries"]:
        print(f"{record['namespaces'][0]}: {record['content']['text']}")

Now start a new conversation and ask "What needs me?". The Copilot's first card is its look at what it remembers about Dana, and the answer follows her preference. Sign in as Marcus Oyelaran and ask the same question: his namespace is empty, and Dana's notes are never used for him.

## Leave it running for Lab 2

Everything you deployed today stays up for Lab 2, the final teardown, which deletes it all and checks what's left on your account. Keep an eye on your spend with `uv run monitor.py`.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Build your own LangGraph solution on AgentCore Runtime, for a business problem where agents make decisions within limits, like approving a claim. Give it decisions with limits, where a person signs off above the limit; durable state, with LangGraph checkpoints in a database; an enterprise context layer for the agent; AgentCore Memory with a namespace per user; and deploy it on AgentCore Runtime. It can be a variation of UnderCover or something completely different.<br/><br/>I'd love it if you would consider making a PR with a short write-up of your solution and a link to your github in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>